In [5]:
"""
TUTORIAL 1: Simple Conversational Agent
Using existing Gen-AI project structure
"""

import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent  # Go up from notebooks/ to Gen-AI/
sys.path.insert(0, str(project_root))

# Use your existing env_loader
from utils.env_loader import load_environment

# Load environment
load_environment()

print("="*80)
print("TUTORIAL 1: SIMPLE CONVERSATIONAL AGENT")
print("="*80)
print(f"Project root: {project_root}")
print(f"Working from: {Path.cwd()}")

TUTORIAL 1: SIMPLE CONVERSATIONAL AGENT
Project root: /Users/kanderaolaxminarasimharao/Downloads/MyAgents-Git/MyAgents
Working from: /Users/kanderaolaxminarasimharao/Downloads/MyAgents-Git/MyAgents/notebooks


In [ ]:
#import os
#tavily_key = os.getenv("TAVILY_API_KEY")  # works as long as it's in .env

# TUTORIAL 3: TASKIFIER — INTELLIGENT TASK PLANNING AGENT
 
**What You'll Learn:**
 
✅ Custom TypedDict State — state beyond just messages \
✅ Sequential multi-node pipelines — nodes passing enriched state forward \
✅ Tavily Web Search — real-time internet retrieval as a tool \
✅ ChatPromptTemplate.from_template — simpler single-string prompts \
✅ Graph visualization — drawing your pipeline with Mermaid \
✅ Stateless pipelines — when you don't need memory
 
# **Key Components** 
Custom State: Holds task, style, details and plan — not just messages \
Approach Analysis Node: Reads past work history and infers user style \
Task Knowledge Retrieval Node: Searches the web for task-specific information \
Customized Approach Generation Node: Combines everything into a personalized plan
 
# **Method Details** 
1. Setup Environment 
2. Define Custom State 
3. Initialize LLM and Tavily 
4. Build Three Specialized Nodes 
5. Wire Nodes into a Sequential Graph 
6. Run the Agent and Inspect Output
 

In [11]:
import os
from typing import TypedDict
 
# LangGraph
from langgraph.graph import StateGraph, END
from langchain_core.runnables.graph import MermaidDrawMethod
 
# LangChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
 
# Tavily — web search tool
from tavily import TavilyClient
 
# Display
from IPython.display import display, Image
 
print("✅ All imports successful!")
print("\n📦 New libraries:")
print("  • TypedDict — define custom state fields")
print("  • TavilyClient — real-time web search")
print("  • MermaidDrawMethod — graph visualization")
 
print("📖 LESSON 1: Custom State — Beyond Just Messages")
print("="*80)

✅ All imports successful!

📦 New libraries:
  • TypedDict — define custom state fields
  • TavilyClient — real-time web search
  • MermaidDrawMethod — graph visualization
📖 LESSON 1: Custom State — Beyond Just Messages


In [10]:
#!pip install tavily

In [12]:
class ApproachState(TypedDict):
    """
    Custom state for the Taskifier agent.
 
    Key Concept:
    - State does NOT have to be messages!
    - You can define any fields your pipeline needs
    - Each node reads some fields, enriches them, and passes state forward
    """
    task: str     # User's input task — set at the start, never changes
    history: str  # Past work history — loaded by Node 1
    style: str    # Analyzed work style — written by Node 1
    details: str  # Web search results — written by Node 2
    plan: str     # Final personalized plan — written by Node 3
 
print("✓ Custom state defined")
print("\n📝 How state flows through nodes:")
print("""
  ┌──────────────────────────────────────────────────────┐
  │ ApproachState                                        │
  │                                                      │
  │  task     → set by user at start                    │
  │  history  → Node 1 reads from disk                  │
  │  style    → Node 1 writes (LLM analysis)            │
  │  details  → Node 2 writes (Tavily web search)       │
  │  plan     → Node 3 writes (final output)            │
  └──────────────────────────────────────────────────────┘
""")
print("💡 Compare to Tutorial 1 & 2:")
print("  Tutorial 1 & 2: messages: Annotated[Sequence[BaseMessage], add_messages]")
print("  Tutorial 3:     task, history, style, details, plan — all plain strings")
print("\n  No add_messages needed because we are not managing chat history!")
 

✓ Custom state defined

📝 How state flows through nodes:

  ┌──────────────────────────────────────────────────────┐
  │ ApproachState                                        │
  │                                                      │
  │  task     → set by user at start                    │
  │  history  → Node 1 reads from disk                  │
  │  style    → Node 1 writes (LLM analysis)            │
  │  details  → Node 2 writes (Tavily web search)       │
  │  plan     → Node 3 writes (final output)            │
  └──────────────────────────────────────────────────────┘

💡 Compare to Tutorial 1 & 2:
  Tutorial 1 & 2: messages: Annotated[Sequence[BaseMessage], add_messages]
  Tutorial 3:     task, history, style, details, plan — all plain strings

  No add_messages needed because we are not managing chat history!


In [14]:
print("📖 LESSON 2: Setting Up LLM and Tavily")
print("="*80)
 
# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✓ LLM initialized: gpt-4o-mini")
 
# Tavily web search client
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
print("✓ Tavily client initialized")
 
print("\n💡 What is Tavily?")
print("""
  Tavily is a real-time web search API designed for LLM applications.
 
  Tutorial 2 used ChromaDB RAG → searches YOUR documents (offline)
  Tutorial 3 uses Tavily       → searches the LIVE INTERNET (online)
 
  When to use each:
  • ChromaDB RAG → your own knowledge base, private docs
  • Tavily       → current events, public information, live data
""")
 
print("📖 LESSON 3: from_template vs from_messages")
print("="*80)
 
print("""
You have seen ChatPromptTemplate.from_messages() in Tutorial 1 & 2:
 
  prompt = ChatPromptTemplate.from_messages([
      ("system", "You are a helpful assistant."),
      MessagesPlaceholder(variable_name="messages"),   ← full message history
  ])
 
  Use when: You need system/human/assistant structure + conversation history
 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 
Taskifier uses ChatPromptTemplate.from_template():
 
  prompt = ChatPromptTemplate.from_template(
      "Analyze the work style from: {history}"
  )
  formatted = prompt.format(history=approach['history'])
 
  Use when: You just need a single prompt string with variable placeholders
 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 
  from_messages → structured multi-turn chat (history + system prompt)
  from_template → single one-shot prompt with variables
""")
 

📖 LESSON 2: Setting Up LLM and Tavily
✓ LLM initialized: gpt-4o-mini
✓ Tavily client initialized

💡 What is Tavily?

  Tavily is a real-time web search API designed for LLM applications.
 
  Tutorial 2 used ChromaDB RAG → searches YOUR documents (offline)
  Tutorial 3 uses Tavily       → searches the LIVE INTERNET (online)
 
  When to use each:
  • ChromaDB RAG → your own knowledge base, private docs
  • Tavily       → current events, public information, live data

📖 LESSON 3: from_template vs from_messages

You have seen ChatPromptTemplate.from_messages() in Tutorial 1 & 2:
 
  prompt = ChatPromptTemplate.from_messages([
      ("system", "You are a helpful assistant."),
      MessagesPlaceholder(variable_name="messages"),   ← full message history
  ])
 
  Use when: You need system/human/assistant structure + conversation history
 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 
Taskifier uses ChatPromptTemplate.from_template():
 
  prompt = ChatPromptTemplate.from_template(


In [ ]:
print("📖 LESSON 4: Node 1 — Approach Analysis")
print("="*80)
 
def approach_analysis(state: ApproachState) -> ApproachState:
    """
    Node 1: Reads past work history and analyzes user work style.
 
    Flow:
    1. Read all .txt files from the history/ folder
    2. Combine them into a single history string
    3. Ask LLM to analyze and summarize the work style
    4. Write style back to state
 
    Reads from state:  task (not used yet), history (empty)
    Writes to state:   history (loaded), style (analyzed)
    """
    # Load past work history from local files
    history = ""
    history_dir = Path.cwd().parent / "history"
    
    if history_dir.exists():
        for h in os.listdir(history_dir):
            if h.endswith(".txt"):
                with open(history_dir / h) as f:
                    content = f.readlines()
                history = f"{history}\n{content[0]}"
 
    state['history'] = history
 
    # Ask LLM to analyze the work style
    prompt = ChatPromptTemplate.from_template(
        "Analyze the work style the following summary of work history portrays. "
        "Provide a brief summary of the preference in work style."
        "\n\nWork History: {history}"
    )
 
    style = llm.invoke(prompt.format(history=state['history']))
    state['style'] = style
 
    return state
 
print("✓ Node 1 created: approach_analysis")
print("\n💡 What this node does:")
print("  Reads past work history → LLM analyzes style → writes to state")
print("\n💡 Real-world equivalent in your Actor-Critic system:")
print("  This is like the Actor reading the reasoning bank before proposing codes")
 


In [15]:
print("📖 LESSON 5: Node 2 — Task Knowledge Retrieval (Tavily Web Search)")
print("="*80)
 
def task_manifest(state: ApproachState) -> ApproachState:
    """
    Node 2: Searches the web for information about the task.
 
    Flow:
    1. Build a search query from the task
    2. Call Tavily API with max_results=10
    3. Concatenate all result content into a single string
    4. Write details back to state
 
    Reads from state:  task
    Writes to state:   details
    """
    search_query = f"What are the steps for the following task? {state['task']}"
 
    searches = tavily_client.search(search_query, max_results=10)
 
    details = ""
    for result in searches['results']:
        details = details + " " + result['content'] if details else result['content']
 
    state['details'] = details
    return state
 
print("✓ Node 2 created: task_manifest")
print("\n💡 What this node does:")
print("  Reads task → searches live web via Tavily → writes details to state")

📖 LESSON 5: Node 2 — Task Knowledge Retrieval (Tavily Web Search)
✓ Node 2 created: task_manifest

💡 What this node does:
  Reads task → searches live web via Tavily → writes details to state


In [18]:
print("\n📊 Compare RAG vs Tavily:")
print("""
  Tutorial 2 RAG (ChromaDB):
  ┌────────────────────────────────────────────┐
  │ Your documents → chunk → embed → search   │
  │ Offline, private, your own content        │
  └────────────────────────────────────────────┘
 
  Tutorial 3 Tavily:
  ┌────────────────────────────────────────────┐
  │ Live internet search → top 10 results     │
  │ Online, public, up-to-date content        │
  └────────────────────────────────────────────┘
""")
 
print("📖 LESSON 6: Node 3 — Customized Approach Generation")
print("="*80)
 
def result_approach(state: ApproachState) -> ApproachState:
    """
    Node 3: Synthesizes everything into a personalized plan.
 
    Flow:
    1. Read task + details + style from state
    2. Build a rich prompt combining all three
    3. Ask LLM to generate a numbered step-by-step plan
    4. Write plan back to state
 
    Reads from state:  task, details, style
    Writes to state:   plan
    """
    prompt = ChatPromptTemplate.from_template(
        "Give me a plan of steps to carry out the following task with custom work styles specified.\n"
        "You have to pay extra attention to Work Style mentioned below and adjust the plan accordingly.\n\n"
        "Task: {task}\n\n"
        "Details: {details}\n\n"
        "Work Style: {style}\n\n"
        "The output must be a numbered list of steps with explanation of "
        "why it is needed, what to do and how it considers the Work Style."
    )
 
    suggestion = llm.invoke(
        prompt.format(
            task=state['task'],
            details=state['details'],
            style=state['style']
        )
    )
 
    state['plan'] = suggestion
    return state

print("✓ Node 3 created: result_approach")
print("\n💡 What this node does:")
print("  Reads task + details + style → synthesizes → writes plan to state")
print("\n💡 This is the synthesis step — all previous node outputs come together here")
print("   task (user input) + details (web) + style (history) → personalized plan")
 


📊 Compare RAG vs Tavily:

  Tutorial 2 RAG (ChromaDB):
  ┌────────────────────────────────────────────┐
  │ Your documents → chunk → embed → search   │
  │ Offline, private, your own content        │
  └────────────────────────────────────────────┘
 
  Tutorial 3 Tavily:
  ┌────────────────────────────────────────────┐
  │ Live internet search → top 10 results     │
  │ Online, public, up-to-date content        │
  └────────────────────────────────────────────┘

📖 LESSON 6: Node 3 — Customized Approach Generation
✓ Node 3 created: result_approach

💡 What this node does:
  Reads task + details + style → synthesizes → writes plan to state

💡 This is the synthesis step — all previous node outputs come together here
   task (user input) + details (web) + style (history) → personalized plan


In [ ]:
print("📖 LESSON 7: Building the Sequential Graph")
print("="*80)
 
# Initialize graph with our custom state
workflow = StateGraph(ApproachState)
 
# Add three nodes
workflow.add_node("approach_analysis", approach_analysis)
workflow.add_node("task_knowledge_retrieval", task_manifest)
workflow.add_node("customized_approach_generation", result_approach)
 
# Wire them sequentially — fixed edges, no conditions
workflow.set_entry_point("approach_analysis")
workflow.add_edge("approach_analysis", "task_knowledge_retrieval")
workflow.add_edge("task_knowledge_retrieval", "customized_approach_generation")
workflow.add_edge("customized_approach_generation", END)
 
# Compile — no checkpointer needed (stateless pipeline)
app = workflow.compile()
 
print("✓ Graph compiled")
print("\n📊 Graph Structure:")
print("""
  START
    ↓
  [approach_analysis]          ← Node 1: reads history, writes style
    ↓
  [task_knowledge_retrieval]   ← Node 2: searches web, writes details
    ↓
  [customized_approach_generation]  ← Node 3: combines all, writes plan
    ↓
  END
""")
print("💡 Key differences from Tutorial 1 & 2:")
print("  Tutorial 1: START → agent → END (1 node, with memory)")
print("  Tutorial 2: START → agent ↔ tools → END (conditional loop)")
print("  Tutorial 3: START → node1 → node2 → node3 → END (sequential, no memory)")
 
print("📖 LESSON 8: Visualizing the Graph")
print("="*80)
print("LangGraph can draw your pipeline as a diagram — very useful for debugging!")
print()
 
display(
    Image(
        app.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)
 
print("\n💡 Use this whenever your graph gets complex")
print("   It visually confirms your nodes and edges are connected correctly")
 
